<a href="https://colab.research.google.com/github/ingdia/summative-intro-ml/blob/main/credit_default_risk_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
uciml_default_of_credit_card_clients_dataset_path = kagglehub.dataset_download('uciml/default-of-credit-card-clients-dataset')

print('Data source import complete.')


Using Colab cache for faster access to the 'default-of-credit-card-clients-dataset' dataset.
Data source import complete.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Scikit-Learn Imports
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

# TensorFlow / Keras Imports
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Libraries loaded successfully. TensorFlow version:", tf.__version__)

Libraries loaded successfully. TensorFlow version: 2.21.0


In [3]:
# Load dataset from Kaggle directory (adjust path if necessary based on your added dataset folder name)
path = '../input/default-of-credit-card-clients-dataset/UCI_Credit_Card.csv'
if not os.path.exists(path):
    # Fallback search path
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if filename.endswith('.csv'):
                path = os.path.join(dirname, filename)

df = pd.read_csv(path)
print(f"Dataset shape: {df.shape}")

# Rename target column for convenience if needed
if 'default.payment.next.month' in df.columns:
    df.rename(columns={'default.payment.next.month': 'target'}, inplace=True)
elif 'default payment next month' in df.columns:
    df.rename(columns={'default payment next month': 'target'}, inplace=True)

print("Class distribution:")
print(df['target'].value_counts(normalize=True))

Dataset shape: (30000, 25)
Class distribution:
target
0    0.7788
1    0.2212
Name: proportion, dtype: float64


In [4]:
# Drop ID column if present
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# Feature Engineering: Create Total Bill Amount and Total Payment Amount ratios
df['TOTAL_BILL'] = df[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].sum(axis=1)
df['TOTAL_PAY'] = df[['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].sum(axis=1)
df['PAY_TO_BILL_RATIO'] = df['TOTAL_PAY'] / (df['TOTAL_BILL'] + 1) # avoid division by zero

X = df.drop(columns=['target'])
y = df['target'].values

# Train-Test Split (80% train/val, 20% test)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Further split train into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=SEED, stratify=y_train_val
)

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Validation set shape: {X_val_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

Training set shape: (19200, 26)
Validation set shape: (4800, 26)
Test set shape: (6000, 26)


In [5]:
# Master list to store experiment tracking results
experiment_records = []

def log_experiment(model_name, hyperparameters, train_acc, val_acc, precision, recall, f1, auc):
    experiment_records.append({
        "Model": model_name,
        "Hyperparameters": str(hyperparameters),
        "Train Acc": round(train_acc, 4),
        "Val Acc": round(val_acc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(auc, 4)
    })

In [6]:
# Experiment 1: Logistic Regression (Baseline)
lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_val_scaled)
y_prob_lr = lr.predict_proba(X_val_scaled)[:, 1]
log_experiment("Logistic Regression", {"C": 1.0, "solver": "lbfgs"},
               lr.score(X_train_scaled, y_train), accuracy_score(y_val, y_pred_lr),
               precision_score(y_val, y_pred_lr), recall_score(y_val, y_pred_lr),
               f1_score(y_val, y_pred_lr), roc_auc_score(y_val, y_prob_lr))

# Experiment 2: Support Vector Machine (RBF Kernel)
svm_model = SVC(kernel='rbf', probability=True, random_state=SEED)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_val_scaled)
y_prob_svm = svm_model.predict_proba(X_val_scaled)[:, 1]
log_experiment("SVM (RBF)", {"kernel": "rbf", "C": 1.0},
               svm_model.score(X_train_scaled, y_train), accuracy_score(y_val, y_pred_svm),
               precision_score(y_val, y_pred_svm), recall_score(y_val, y_pred_svm),
               f1_score(y_val, y_pred_svm), roc_auc_score(y_val, y_prob_svm))

# Experiment 3: Random Forest (Baseline)
rf_base = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_base.fit(X_train_scaled, y_train)
y_pred_rf1 = rf_base.predict(X_val_scaled)
y_prob_rf1 = rf_base.predict_proba(X_val_scaled)[:, 1]
log_experiment("Random Forest (Baseline)", {"n_estimators": 100},
               rf_base.score(X_train_scaled, y_train), accuracy_score(y_val, y_pred_rf1),
               precision_score(y_val, y_pred_rf1), recall_score(y_val, y_pred_rf1),
               f1_score(y_val, y_pred_rf1), roc_auc_score(y_val, y_prob_rf1))

# Experiment 4: Random Forest (Tuned Depth & Estimators)
rf_tuned = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, random_state=SEED)
rf_tuned.fit(X_train_scaled, y_train)
y_pred_rf2 = rf_tuned.predict(X_val_scaled)
y_prob_rf2 = rf_tuned.predict_proba(X_val_scaled)[:, 1]
log_experiment("Random Forest (Tuned)", {"n_estimators": 200, "max_depth": 10},
               rf_tuned.score(X_train_scaled, y_train), accuracy_score(y_val, y_pred_rf2),
               precision_score(y_val, y_pred_rf2), recall_score(y_val, y_pred_rf2),
               f1_score(y_val, y_pred_rf2), roc_auc_score(y_val, y_prob_rf2))

# Experiment 5: Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=SEED)
gb_model.fit(X_train_scaled, y_train)
y_pred_gb = gb_model.predict(X_val_scaled)
y_prob_gb = gb_model.predict_proba(X_val_scaled)[:, 1]
log_experiment("Gradient Boosting", {"n_estimators": 150, "lr": 0.05, "max_depth": 5},
               gb_model.score(X_train_scaled, y_train), accuracy_score(y_val, y_pred_gb),
               precision_score(y_val, y_pred_gb), recall_score(y_val, y_pred_gb),
               f1_score(y_val, y_pred_gb), roc_auc_score(y_val, y_prob_gb))

print("Traditional ML experiments completed!")

Traditional ML experiments completed!


In [ ]:
from tensorflow.keras.layers import Input

# Build tf.data pipeline
batch_size = 64
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_scaled, y_train))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_scaled, y_val))
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Experiment 6: Deep Learning Sequential Model (Basic Architecture)
model_dl1 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_dl1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history1 = model_dl1.fit(train_dataset, validation_data=val_dataset, epochs=15, verbose=0)

y_prob_dl1 = model_dl1.predict(X_val_scaled).ravel()
y_pred_dl1 = (y_prob_dl1 >= 0.5).astype(int)
log_experiment("Deep Learning (Basic)", {"epochs": 15, "layers": [64, 32]},
               history1.history['accuracy'][-1], history1.history['val_accuracy'][-1],
               precision_score(y_val, y_pred_dl1), recall_score(y_val, y_pred_dl1),
               f1_score(y_val, y_pred_dl1), roc_auc_score(y_val, y_prob_dl1))

# Experiment 7: Deep Learning Optimized Model (Dropout, BatchNorm, Early Stopping)
model_dl2 = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_dl2.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history2 = model_dl2.fit(train_dataset, validation_data=val_dataset, epochs=25, callbacks=[early_stop], verbose=0)

y_prob_dl2 = model_dl2.predict(X_val_scaled).ravel()
y_pred_dl2 = (y_prob_dl2 >= 0.5).astype(int)
log_experiment("Deep Learning (Optimized + Dropout)", {"epochs": 25, "lr": 0.001, "dropout": 0.3},
               history2.history['accuracy'][-1], history2.history['val_accuracy'][-1],
               precision_score(y_val, y_pred_dl2), recall_score(y_val, y_pred_dl2),
               f1_score(y_val, y_pred_dl2), roc_auc_score(y_val, y_prob_dl2))

# Display Master Experiment Table
results_df = pd.DataFrame(experiment_records)
display(results_df)

In [ ]:
# Evaluate best model on final Test set
best_model = gb_model
y_test_pred = best_model.predict(X_test_scaled)
y_test_prob = best_model.predict_proba(X_test_scaled)[:, 1]

print("=== FINAL TEST SET PERFORMANCE ===")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Best Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Save models locally using modern formats
os.makedirs('saved_models', exist_ok=True)
joblib.dump(best_model, 'saved_models/best_traditional_model.joblib')
model_dl2.save('saved_models/best_deep_learning_model.keras')
print("Models successfully saved to 'saved_models/' folder!")